In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 5
fig_height = 4
fig_format = 'png'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'RDpcUmVwb3NpdG9yaWVzXEFENjk4LWdlbmVyYXRpdmUtYWktZm9yLUJB'
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"C:\\Users\\nakul\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\Lib\\importlib\\_bootstrap.py": 1772917703.8481817, "C:\\Users\\nakul\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\Lib\\importlib\\_bootstrap_external.py": 1772917703.8490603, "C:\\Users\\nakul\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\Lib\\zipimport.py": 1772917704.7296336, "C:\\Users\\nakul\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\Lib\\codecs.py": 1772917703.4405446, "C:\\Users\\nakul\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\Lib\\encodings\\aliases.py": 1772917703.512063, "C:\\Users\\nakul\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\Lib\\encodings\\__init__.py": 1772917703.5063918, "C:\\Users\\nakul\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\Lib\\encodings\\utf_8.py": 1772917703.5772903, "C:\\Users\\nakul\\AppData\\Roaming\\uv\\python\\cpython-3.12.

In [2]:
#| echo: false
#| eval: false
#| output: markdown

from datetime import datetime, timedelta
import pandas as pd
from IPython.display import display, Markdown

# Function to generate the Day 1 - Day 7 list based on a start date
def generate_days(start_date):
    days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    start_day_index = days.index(start_date)
    days_shifted = days[start_day_index:] + days[:start_day_index]
    return days_shifted

# Start dates for each section
start_dates = {
    "A1": "Tuesday",
    "A2": "Thursday",
    "A3": "Wednesday"
}

# Initialize the combined table data
combined_data = {"Day": [f"Day {i+1}" for i in range(7)]}

# Generate and add columns for each section
for section, start_date in start_dates.items():
    combined_data[section] = generate_days(start_date)

# Create a single DataFrame
combined_table = pd.DataFrame(combined_data)

# Display the combined table without index
display(combined_table.style.hide(axis="index"))

In [3]:
#| label: build-course-schedule
#| echo: false
#| warning: false
#| output: html

import pandas as pd
from IPython.display import display, HTML
import sys

# -------------------------------------------------
# Clear module cache (important for Quarto reruns)
# -------------------------------------------------
for m in list(sys.modules):
    if m.startswith(("schedules", "bu_calendar")):
        del sys.modules[m]

from schedules.semester_planner import build_section_schedules, get_lecture_catalog
from schedules.config import semester, year, start_dates_by_semester

# =================================================
# USER INPUT (only edit this section)
# =================================================
modalities = ["oncampus", "o1", "o2"]

section_plan = build_section_schedules(
    semester=semester,
    year=year,
    modalities=modalities,
    start_dates_by_semester=start_dates_by_semester,
)

# -------------------------------------------------
# Load COURSE DESCRIPTION (content only)
# -------------------------------------------------
course_table = (
    pd.read_excel(
        "data/AD698-Schedule.xlsx",
        sheet_name="Course Details"
    )
)

module0_catalog = (
    course_table
    .loc[
        course_table["Lecture"].astype(str).str.strip().str.upper().str.match(r"^L0\.\d+$", na=False),
        ["Lecture", "Title", "Description"],
    ]
    .reset_index(drop=True)
)

course_table = course_table.drop(columns=["Modules", "Module Name"], errors="ignore")

# Keep lecture rows only, excluding Module 0
course_table = get_lecture_catalog(course_table)
max_lectures = max(p.lecture_count for p in section_plan.values())
course_table = course_table.head(max_lectures).reset_index(drop=True)
course_table = course_table[["Lecture", "Title"]]
# -------------------------------------------------
# Generate schedules PER SECTION
# -------------------------------------------------
section_dates = {
    section: plan.lecture_dates for section, plan in section_plan.items()
}

# -------------------------------------------------
# Attach section columns (ONLINE LAST)
# -------------------------------------------------
oncampus_sections = [
    s for s, p in section_plan.items() if p.modality == "oncampus"
]
online_sections = [
    s for s, p in section_plan.items() if p.modality != "oncampus"
]

# On-campus first
for s in oncampus_sections:
    dates = section_dates[s]
    course_table[s] = dates + [pd.NaT] * (max_lectures - len(dates))

# Online last
for s in online_sections:
    dates = section_dates[s]
    course_table[s] = dates + [pd.NaT] * (max_lectures - len(dates))

# -------------------------------------------------
# Date formatting
# -------------------------------------------------


def fmt(d):
    return "-" if pd.isna(d) else d.strftime("%d-%b")


for col in course_table.columns[2:]:
    course_table[col] = course_table[col].map(fmt)

module0_table = pd.DataFrame([
    {
        "Section": section,
        "Module 0 Opens": plan.module0_open.strftime("%d-%b (%a)"),
        "First Lecture": plan.lecture_dates[0].strftime("%d-%b (%a)"),
    }
    for section, plan in section_plan.items()
])

module0_preview_styled = (
    module0_catalog
    .style
    .hide(axis="index")
    # .set_caption("Week Ahead: Module 0 Preview")
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]},
        {"selector": "td", "props": [
            ("text-align", "left"),
            ("vertical-align", "top"),
            ("white-space", "normal")
        ]},
    ])
)

# -------------------------------------------------
# Presentation
# -------------------------------------------------
module0_styled = (
    module0_table
    .style
    .hide(axis="index")
    # .set_caption("Module 0 Release Plan (opens 10 days before first class)")
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]},
        {"selector": "td", "props": [
            ("text-align", "left"),
            ("vertical-align", "top"),
            ("white-space", "nowrap")
        ]},
    ])
)

styled = (
    course_table
    .style
    .hide(axis="index")
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]},
        {"selector": "td", "props": [
            ("text-align", "left"),
            ("vertical-align", "top"),
            ("white-space", "nowrap")
        ]},
    ])
)

course_table.drop(columns=["A1","O2"], inplace=True)
module0_table.drop(index=[0,2], inplace=True)
display(HTML(module0_styled.to_html()))
display(HTML(module0_preview_styled.to_html()))
display(HTML(styled.to_html()))

Section,Module 0 Opens,First Lecture
O1,25-Apr (Sat),05-May (Tue)


Lecture,Title,Description
L0.1,Colab + GPU + Tensors,"Compute setup, tensors, GPU execution"
L0.2,"Math for Neural Networks, Regression and Classification Problems","NN, loss, gradients, training loop"
L0.3,Sequence Learning Problem,"Temporal modeling, intro to sequences"


Lecture,Title,O1
L1.1,Course Orientation and Reproducible GenAI Setup,05-May
L1.2,Language Probability and Generative Systems,07-May
L2.1,Mathematical Foundations Through Language Modeling,12-May
L2.2,From Words to Structured Text,14-May
L3.1,Prompting as System Design,19-May
L3.2,Tokenization Embeddings and Semantic Geometry,21-May
L4.1,Transformers Attention and Context,26-May
L4.2,Training Paradigms and Fine-Tuning Pipelines,28-May
L5.1,Retrieval Augmented Generation Concepts,02-Jun
L5.2,Designing Retrieval-Augmented Pipelines,04-Jun
